# Base brasileira — grade de cinco minutos e origem

Resolução real de **cinco minutos**, horizonte **t+5 minutos**. Os nomes dos arquivos foram mantidos por solicitação do usuário; o conteúdo foi migrado de diário para intradiário.

O alvo e a persistência (t−5 min) precisam ser originalmente medidos. Cada etapa de preenchimento causal usa até 15 minutos; no Brasil a combinação proxy + ffill pode atingir 30 minutos somente no histórico: nunca se interpola usando ponto posterior. Não se concatenam blocos interrompidos. A avaliação principal é diurna (elevação solar > 0); a avaliação 24h é complementar. Zero noturno não deve inflar a precisão.

As entradas principais são potência AC em p.u., hora local e dia do ano em seno/cosseno. O vento continua excluído. ERA5 nativa horária é auxiliar e não entra nos modelos principais. Estatísticas de normalização vêm apenas do treino australiano; alvos de validação/teste não ajustam escalas.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Image,Markdown
RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/energia_transfer').is_dir())
if str(RAIZ/'src') not in sys.path: sys.path.insert(0,str(RAIZ/'src'))
PASTA=RAIZ/'experimentos/transfer_learning_australia_brasil'
AU=RAIZ/'experimentos/australia_deep_learning'
REPORT=RAIZ/'reports/intradiario'
from energia_transfer.transfer_learning.dados import carregar,particao
_,CONFIG,CONTRATO=carregar(PASTA)
def exibir(t): display(t.round(5))


In [2]:
df=pd.read_pickle(RAIZ/'data/processed/brasil/base_5min_brasil.pkl')
exibir(df.head(15))
print('Alvos medidos:',df.Active_Power_original.notna().sum())
print('Históricos reconstruídos:',df.potencia_estimada.sum())
print('Grade:',df.index.to_series().diff().dropna().value_counts().to_dict())


Alvos medidos: 90023
Históricos reconstruídos: 12627
Grade: {Timedelta('0 days 00:05:00'): 105119}


,Active_Power_original,Active_Power,registro_original,diurno,inversores_medidos,inversores_proxy,inversores_zero_fisico,potencia_estimada,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,hora_sin,hora_cos,dia_ano_sin,dia_ano_cos,meteo_resolucao_nativa_min,meteo_idade_min,temperatura_estacao_original,ghi_estacao_original_escala_nao_validada
timestamp,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.00000,1.00000,0.0,1.0,60,0,NaN,NaN
2025-01-01 00:05:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.02181,0.99976,0.0,1.0,60,5,NaN,NaN
2025-01-01 00:10:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.04362,0.99905,0.0,1.0,60,10,NaN,NaN
2025-01-01 00:15:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.06540,0.99786,0.0,1.0,60,15,NaN,NaN
2025-01-01 00:20:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.08716,0.99619,0.0,1.0,60,20,NaN,NaN
2025-01-01 00:25:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.10887,0.99406,0.0,1.0,60,25,NaN,NaN
2025-01-01 00:30:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.13053,0.99144,0.0,1.0,60,30,NaN,NaN
2025-01-01 00:35:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.15212,0.98836,0.0,1.0,60,35,NaN,NaN
2025-01-01 00:40:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.17365,0.98481,0.0,1.0,60,40,NaN,NaN


In [3]:
exibir(pd.read_csv(PASTA/'resultados/particoes.csv'))
print('Inputs:',CONTRATO['variaveis_por_passo'])
print('Históricos:',[l*5 for l in CONFIG['lags']],'minutos; horizonte: 5 minutos')
print(CONTRATO['normalizacao'])


Inputs: ['Active_Power', 'hora_sin', 'hora_cos', 'dia_ano_sin', 'dia_ano_cos']
Históricos: [30, 60] minutos; horizonte: 5 minutos
Potência AC em p.u.; escalas do treino australiano diurno compartilhadas no Brasil.


,dominio,lag,historico_min,particao,janelas_antes_coorte,janelas,primeiro_alvo,ultimo_alvo
0,australia,6,30,treino,488964,40747,2013-07-25T17:05:00,2022-12-31T18:25:00
1,australia,12,60,treino,488448,40704,2013-07-25T17:35:00,2022-12-31T18:25:00
2,australia,6,30,validacao,46840,46840,2023-01-01T05:55:00,2023-12-31T19:20:00
3,australia,12,60,validacao,46840,46840,2023-01-01T05:55:00,2023-12-31T19:20:00
4,australia,6,30,teste,63393,63393,2024-01-01T06:00:00,2025-08-22T18:15:00
5,australia,12,60,teste,63393,63393,2024-01-01T06:00:00,2025-08-22T18:15:00
6,brasil,6,30,treino,32275,32275,2025-01-01T06:25:00,2025-09-30T18:10:00
7,brasil,12,60,treino,31874,31874,2025-01-01T06:55:00,2025-09-30T18:10:00
8,brasil,6,30,validacao,4043,4043,2025-10-01T06:00:00,2025-10-31T18:15:00
9,brasil,12,60,validacao,4043,4043,2025-10-01T06:00:00,2025-10-31T18:15:00


## Reconstrução causal dos inputs

Para um inversor ausente, o proxy usa os outros sete medidos no instante e a mediana da razão dos últimos 30 dias, excluindo o instante atual e exigindo 100 doadores. Só até três ausências consecutivas já conhecidas. Não consulta o fim futuro da lacuna. Zeros físicos são restritos à escuridão solar e conferidos contra as medições. Ffill usa no máximo três passos e não encadeia estimativas indefinidamente.

`Active_Power_original` é a verdade supervisionada; `Active_Power` pode conter reconstrução apenas para contexto. Na avaliação exigimos também a persistência originalmente medida no instante anterior. Falhas mais longas interrompem as janelas. Não eliminamos um dia inteiro por uma única leitura ausente.
